In [ ]:
import os
import torch
import argparse
import numpy as np
from osgeo import gdal

from Model.DHD_Net.Dual_Task_Fusion_Network import Dual_Task as DHD_Net

os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

In [ ]:
# ------------------------ Arguments ------------------------
parser = argparse.ArgumentParser()
parser.add_argument('--data_dir', type=str, default='Your tif folder path')
parser.add_argument('--out_dir', type=str, default='./Data/Predictions')
parser.add_argument('--weights', type=str, default='./Model_Weight/DHD-Net.pkl')
parser.add_argument('--class_num', type=int, default=6)
parser.add_argument('--img_size', type=int, default=128)
parser.add_argument('--stride', type=int, default=128)
parser.add_argument('--device', type=str, default='cuda:1')
opt = parser.parse_args(args=[])

device = torch.device(opt.device if torch.cuda.is_available() else 'cpu')
os.makedirs(opt.out_dir, exist_ok=True)

model = DHD_Net(num_classes=opt.class_num).to(device)
model.load_state_dict(torch.load(opt.weights, map_location=device), strict=False)
model.eval()

In [ ]:
def predict_one_image(img_path, out_dir):
    src = gdal.Open(img_path, gdal.GA_ReadOnly)
    gt = src.GetGeoTransform()
    proj = src.GetProjection()
    width = src.RasterXSize
    height = src.RasterYSize
    bands = src.RasterCount

    out_path = os.path.join(out_dir, os.path.basename(img_path))
    driver = gdal.GetDriverByName('GTiff')
    out_ds = driver.Create(out_path, width, height, 1, gdal.GDT_Byte, options=['COMPRESS=LZW'])
    out_ds.SetGeoTransform(gt)
    out_ds.SetProjection(proj)

    for y in range(0, height, opt.stride):
        for x in range(0, width, opt.stride):
            win_w = min(opt.img_size, width - x)
            win_h = min(opt.img_size, height - y)

            win_data = src.ReadAsArray(x, y, win_w, win_h)
            if win_data.ndim == 2:
                win_data = win_data[np.newaxis, :, :]

            if win_h < opt.img_size or win_w < opt.img_size:
                padded = np.zeros((bands, opt.img_size, opt.img_size), dtype=win_data.dtype)
                padded[:, :win_h, :win_w] = win_data
                win_tensor = torch.from_numpy(padded).unsqueeze(0).to(device)
            else:
                win_tensor = torch.from_numpy(win_data).unsqueeze(0).to(device)

            with torch.no_grad():
                pred = model(win_tensor)
                if isinstance(pred, (list, tuple)):
                    pred = sum(pred) / len(pred)
                pred_class = pred.argmax(dim=1).cpu().numpy()[0]

            out_band = out_ds.GetRasterBand(1)
            out_band.WriteArray(pred_class[:win_h, :win_w], x, y)

    out_ds.FlushCache()
    out_ds = None
    src = None
    print(f"Saved: {out_path}")

In [ ]:
for tif_file in os.listdir(opt.data_dir):
    if tif_file.endswith('.tif'):
        img_path = os.path.join(opt.data_dir, tif_file)
        predict_one_image(img_path, opt.out_dir)